In [ ]:
N_IMAGES = 25
ZIP_URL = "<PRESIGNED_URL>"
WORK = "/tmp/aic"


In [ ]:
import subprocess, os, time, pathlib, json
os.makedirs(WORK, exist_ok=True)
def sh(c, tail=2):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print("$", c, "\n", "\n".join(r.stdout.splitlines()[-tail:]), flush=True); return r
sh("pip install -q easyocr")
sh("pip install -q paddlepaddle paddleocr")
import paddleocr; print("paddleocr", getattr(paddleocr, "__version__", "?"))


In [ ]:
import zipfile, urllib.request
t0=time.time(); z=f"{WORK}/kf.zip"
urllib.request.urlretrieve(ZIP_URL, z)
print(f"tai {os.path.getsize(z)/1e6:.0f} MB / {time.time()-t0:.0f}s", flush=True)
zf=zipfile.ZipFile(z); mem=[m for m in zf.namelist() if m.lower().endswith(".jpg")]
step=max(1,len(mem)//N_IMAGES)
for m in mem[::step][:N_IMAGES]: zf.extract(m, f"{WORK}/imgs")
paths=sorted(str(p) for p in pathlib.Path(f"{WORK}/imgs").rglob("*.jpg"))
print("anh:",len(paths))


In [ ]:
import easyocr
eo = easyocr.Reader(["vi"], gpu=False)
easy = {p: [(d[1], float(d[2])) for d in eo.readtext(p)] for p in paths}
print("easyocr xong")


In [ ]:
# PaddleOCR doi API nhieu lan giua cac ban; thu lan luot cho den khi chay.
from paddleocr import PaddleOCR
po = None
for kw in ({"lang":"vi"}, {"lang":"vi","use_angle_cls":True}, {}):
    try:
        po = PaddleOCR(**kw); print("khoi tao OK voi", kw); break
    except Exception as e:
        print("that bai", kw, "->", type(e).__name__, str(e)[:120])

def run(p):
    for call in (lambda: po.predict(p), lambda: po.ocr(p)):
        try: return call()
        except Exception: continue
    return None

def parse(r):
    """Rut (text, score) tu bat ky hinh dang tra ve nao cua Paddle."""
    out=[]
    if r is None: return out
    if isinstance(r, dict):
        ts, ss = r.get("rec_texts") or [], r.get("rec_scores") or []
        return list(zip(ts, [float(s) for s in ss]))
    if isinstance(r, list):
        for item in r:
            if isinstance(item, dict):
                ts, ss = item.get("rec_texts") or [], item.get("rec_scores") or []
                out += list(zip(ts, [float(s) for s in ss]))
            elif isinstance(item, list):
                for line in item or []:
                    try: out.append((line[1][0], float(line[1][1])))
                    except Exception: pass
    return out

paddle={}
if po is not None:
    t0=time.time()
    for i,p in enumerate(paths):
        paddle[p] = parse(run(p))
        if (i+1)%10==0: print(f"  paddle {i+1}/{len(paths)}", flush=True)
    print(f"paddle xong {time.time()-t0:.0f}s")


In [ ]:
import numpy as np, unicodedata, re
def summ(d,name):
    sc=[s for v in d.values() for _,s in v]
    if not sc: print(f"{name}: khong co ket qua"); return
    print(f"{name:10s} vung/anh {len(sc)/len(d):5.1f} | tin cay trung vi {np.median(sc):.2f} "
          f"| >=0.8: {sum(1 for s in sc if s>=0.8)/len(sc):.1%} "
          f"| tong ky tu {sum(len(t) for v in d.values() for t,_ in v)}")
summ(easy,"easyocr"); summ(paddle,"paddleocr")

def strip_dia(s):
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("\u0111","d")

# Loi hay gap nhat cua OCR o day la d/đ. Bo dau co gop chung lai khong?
for a,b in [("dường","đường"),("dối","đối"),("dảo","đảo"),("dến","đến"),("dồng","đồng")]:
    print(f"  {a:8s} -> {strip_dia(a):8s} | {b:8s} -> {strip_dia(b):8s} | gop duoc: {strip_dia(a)==strip_dia(b)}")


In [ ]:
for p in paths[:10]:
    print(f"\n====== {p.split('imgs/')[-1]}")
    print("  EASYOCR:")
    for t,s in sorted(easy.get(p,[]), key=lambda x:-x[1])[:6]: print(f"    [{s:.2f}] {t}")
    print("  PADDLE:")
    for t,s in sorted(paddle.get(p,[]), key=lambda x:-x[1])[:6]: print(f"    [{s:.2f}] {t}")


In [ ]:
json.dump({"easyocr":{k.split("imgs/")[-1]:v for k,v in easy.items()},
           "paddleocr":{k.split("imgs/")[-1]:v for k,v in paddle.items()}},
          open("/kaggle/working/ocr_quality.json","w",encoding="utf-8"), ensure_ascii=False, indent=1)
print("da luu")
